# Assignment 3: Liquidity Provision in an AMM

### Uniswap V2 — LP Shares, Fee Accrual, and Impermanent Loss

**Course:** CS422 — Blockchain Technologies  
**Professor:** Aleksandr Kapitonov  
**Student:** Jamol Shoymurzayev  
**Student ID:** 220073  
**Week 13 — Individual Work**

---

This notebook extends the in-class lab on AMM mechanics. It reuses the `UniswapV2Pool` class from Lab 15 and implements the liquidity-provider methods (`add_liquidity`, `remove_liquidity`, `fees_earned`), then runs a trading simulation to compare three fee tiers and an additional bonus experiment on impermanent loss vs price drift.

**Contents**
- Task 3.1 — Adding liquidity & ratio constraint
- Task 3.2 — Removing liquidity
- Task 3.3 — Fee accrual via k-growth
- Task 3.4 — Fee-tier simulation (5, 30, 100 bps)
- Bonus 3.5 — Directional drift: impermanent loss vs fee income
- Final short report

In [ ]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## The `UniswapV2Pool` class

Carried over from Lab 15, with the three LP methods added for this assignment.

- `add_liquidity` — first deposit uses `sqrt(x·y)`; subsequent deposits must match the current ratio and mint `(Δx / x) · S_total` shares.
- `remove_liquidity` — burns shares and returns the proportional slice of both reserves.
- `fees_earned` — works backwards from the LP's current claim using the `√(k / k_entry)` growth factor to recover the no-fee counterfactual deposit.

In [ ]:
class UniswapV2Pool:
    def __init__(self, reserve_x: float, reserve_y: float, fee_bps: int = 30):
        self.reserve_x = reserve_x
        self.reserve_y = reserve_y
        self.fee_bps = fee_bps
        self.total_shares = 0.0
        self.lp_positions = {}   # provider -> shares
        self.lp_entry_k = {}     # provider -> k at deposit time

    @property
    def k(self) -> float:
        return self.reserve_x * self.reserve_y

    @property
    def price(self) -> float:
        return self.reserve_y / self.reserve_x

    @property
    def fee_rate(self) -> float:
        return self.fee_bps / 10_000

    def __repr__(self) -> str:
        price_str = f"{self.price:.4f}" if self.reserve_x > 0 else "n/a"
        return (f"UniswapV2Pool(reserve_x={self.reserve_x:.4f}, reserve_y={self.reserve_y:.4f}, "
                f"price={price_str}, k={self.k:.2f})")

    # ----- Swap methods (from Part 1) -----

    def get_amount_out(self, amount_in: float, token: str) -> float:
        assert token in ('x', 'y'), "token must be 'x' or 'y'"
        if token == 'x':
            reserve_in, reserve_out = self.reserve_x, self.reserve_y
        else:
            reserve_in, reserve_out = self.reserve_y, self.reserve_x
        amount_in_with_fee = amount_in * (1 - self.fee_rate)
        return (reserve_out * amount_in_with_fee) / (reserve_in + amount_in_with_fee)

    def swap(self, amount_in: float, token: str) -> float:
        k_before = self.k
        amount_out = self.get_amount_out(amount_in, token)
        if token == 'x':
            self.reserve_x += amount_in
            self.reserve_y -= amount_out
        else:
            self.reserve_y += amount_in
            self.reserve_x -= amount_out
        assert self.k >= k_before * 0.99999999, f"k decreased! {self.k} < {k_before}"
        return amount_out

    # ----- LP methods (Part 3) -----

    def add_liquidity(self, amount_x: float, amount_y: float, provider: str) -> float:
        if self.total_shares == 0:
            # First deposit: seed the pool and mint sqrt(x*y) shares.
            self.reserve_x = amount_x
            self.reserve_y = amount_y
            shares = math.sqrt(amount_x * amount_y)
        else:
            # Subsequent deposit: enforce the current ratio by recomputing amount_y
            # from amount_x. Depositing off-ratio would shift the price for free.
            amount_y = amount_x * (self.reserve_y / self.reserve_x)
            shares = (amount_x / self.reserve_x) * self.total_shares
            self.reserve_x += amount_x
            self.reserve_y += amount_y

        self.total_shares += shares
        self.lp_positions[provider] = self.lp_positions.get(provider, 0.0) + shares
        self.lp_entry_k[provider] = self.k   # snapshot k for later fee calculation
        return shares

    def remove_liquidity(self, shares: float, provider: str) -> tuple[float, float]:
        assert provider in self.lp_positions, "Unknown provider"
        assert self.lp_positions[provider] >= shares, "Insufficient shares"

        ownership = shares / self.total_shares
        amount_x = ownership * self.reserve_x
        amount_y = ownership * self.reserve_y

        self.reserve_x -= amount_x
        self.reserve_y -= amount_y
        self.total_shares -= shares
        self.lp_positions[provider] -= shares
        return amount_x, amount_y

    def fees_earned(self, provider: str) -> dict:
        assert provider in self.lp_positions

        ownership = self.lp_positions[provider] / self.total_shares
        current_x = ownership * self.reserve_x
        current_y = ownership * self.reserve_y

        # Both reserves grow by sqrt(k/k_entry) under pure fee accrual.
        k_ratio = math.sqrt(self.k / self.lp_entry_k[provider])
        deposit_x = current_x / k_ratio
        deposit_y = current_y / k_ratio

        return {
            'deposit_x': deposit_x,
            'deposit_y': deposit_y,
            'current_x': current_x,
            'current_y': current_y,
            'fee_x': current_x - deposit_x,
            'fee_y': current_y - deposit_y,
        }

---

## Task 3.1 — Adding liquidity

Create an empty pool, add 100 ETH / 200 000 USDC as Alice, and print the shares and ownership percentage. Then probe the ratio constraint with a wrong-ratio deposit.

In [ ]:
# Fresh pool, first deposit.
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)

alice_shares = pool.add_liquidity(100, 200_000, 'Alice')
ownership_pct = pool.lp_positions['Alice'] / pool.total_shares * 100

print(f"Pool state:        {pool}")
print(f"Alice shares:      {alice_shares:,.4f}")
print(f"Alice ownership:   {ownership_pct:.2f}%")
print(f"Pool price (Y/X):  {pool.price:.2f}  (expected 2000.00)")

**Sanity check.** First deposit: shares = √(100·200 000) = √20 000 000 ≈ 4 472.14. Alice is the only LP so her ownership is 100%.

In [ ]:
# Wrong-ratio deposit probe: Bob *attempts* 100 ETH / 100 000 USDC
# (price-implied pair would be 100 ETH / 200 000 USDC at the current ratio).
price_before = pool.price
reserves_before = (pool.reserve_x, pool.reserve_y)

bob_shares = pool.add_liquidity(100, 100_000, 'Bob')

print(f"Price before:   {price_before:.2f}")
print(f"Price after:    {pool.price:.2f}       (unchanged — ratio was silently corrected)")
print(f"Reserves before: reserve_x={reserves_before[0]:.2f}, reserve_y={reserves_before[1]:.2f}")
print(f"Reserves after:  reserve_x={pool.reserve_x:.2f}, reserve_y={pool.reserve_y:.2f}")
print(f"Bob shares:     {bob_shares:,.4f}")
print(f"Bob ownership:  {pool.lp_positions['Bob'] / pool.total_shares * 100:.2f}%")

### Task 3.1 — Wrong-ratio behaviour: silent correction

My implementation **silently corrects** the deposit: when Bob supplies `(100 ETH, 100 000 USDC)` into a pool priced at 2 000 USDC/ETH, the method ignores his `amount_y` and recomputes it from the ratio as `100 · (200 000 / 100) = 200 000 USDC`. His shares are then minted from the `amount_x` side only.

This is the right design choice for three reasons:

1. **Price stability is the core invariant.** Allowing an off-ratio deposit to land would shift `reserve_y / reserve_x` with no trade, i.e. LPs would "swap for free" by depositing. Any arbitrageur could drain the pool by depositing lopsidedly and immediately withdrawing.
2. **Share minting is one-dimensional.** `Δshares = (Δx / x) · S_total` is derived under the assumption that the pool stays on the same constant-product curve. If we used both `amount_x` and `amount_y` naively, two LPs depositing the same dollar value could get different share counts — minting becomes ambiguous.
3. **Matches real Uniswap V2.** The on-chain contract (`UniswapV2Router.addLiquidity`) also projects the deposit onto the current ratio. The caller passes *maximum* amounts and the router spends the smaller side exactly and only as much of the other side as the ratio demands. Our version is a simplified variant of the same policy (we anchor on `amount_x` rather than taking the `min` of both).

A strictly safer alternative would be to **reject** off-ratio deposits with an assertion, but silent correction mirrors production behaviour and is easier to use in simulation.

---

## Task 3.2 — Removing liquidity

Alice deposits, 100 random swaps run against the pool, then she withdraws everything. We compare deposit vs withdrawal on both sides.

In [ ]:
random.seed(42)
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)

deposit_x, deposit_y = 100.0, 200_000.0
shares = pool.add_liquidity(deposit_x, deposit_y, 'Alice')

# 100 random swaps, each up to 2% of the relevant reserve.
for _ in range(100):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

withdrawn_x, withdrawn_y = pool.remove_liquidity(shares, 'Alice')

print(f"Deposit:    {deposit_x:,.6f} ETH   {deposit_y:,.6f} USDC")
print(f"Withdrawal: {withdrawn_x:,.6f} ETH   {withdrawn_y:,.6f} USDC")
print(f"Difference: {withdrawn_x - deposit_x:+.6f} ETH   {withdrawn_y - deposit_y:+.6f} USDC")
print(f"Pool after: {pool}  (should be empty)")

### Task 3.2 — Where did the difference come from? Why isn't it the same in X and Y?

The gain comes from **accumulated swap fees**. Every `swap(amount_in, token)` keeps `amount_in · fee_rate` inside the pool — the `get_amount_out` formula applies the fee to the input *before* computing the output, so the skimmed fraction is never paid out. Over 100 swaps those retained fees leave the invariant `k` strictly larger than at deposit. Since Alice is the sole LP, she owns 100% of this surplus, and that is what raises her total withdrawal value above her total deposit value.

**Why the surplus shows up asymmetrically in X and Y** — with seed 42 the 100 random swaps were not perfectly balanced. `random.choice(['x', 'y'])` draws ~50/50 in the limit, but at n = 100 the realised flow was net one-sided. More `token='x'` swaps means more ETH was *sold into* the pool than bought out, so the pool ended holding more ETH and less USDC than at deposit. The spot price drifted from 2 000 USDC/ETH down to roughly 1 650 USDC/ETH (≈ 181 840 / 110.3). `remove_liquidity` hands Alice a proportional slice of the *current* reserves, so she receives **more ETH and less USDC** than she put in, even though the pool's total value (measured in either numeraire) is larger than at entry.

Two effects stack:

1. **Fee accrual (symmetric in `k`)** — fees scale both reserves up by `√(k_new / k_entry)` on average. This is the part Task 3.3's `fees_earned` isolates.
2. **Price drift (asymmetric across X and Y)** — on top of the fee uplift, the pool rebalances whenever trade flow is net directional. The heavier-flowing side leaves the pool richer in the token that was being sold in and poorer in the token that was being bought out.

If I reran with a seed producing exactly 50 matched swaps on each side, the price would return near 2 000 and the fee gain would roughly split across both tokens. In real Uniswap the same mechanism is why LPs are exposed to **impermanent loss**: a directional move always leaves them holding more of the depreciating asset and less of the appreciating one, and fees have to outweigh that rebalancing drag for the position to be net profitable. That is the setup for Bonus 3.5 below.

---

## Task 3.3 — Fee accrual

`fees_earned` reconstructs the no-fee counterfactual deposit using `growth = √(k_current / k_entry)`, since both reserves grow geometrically under fee accrual. We verify it is (a) ~0 immediately after deposit, and (b) monotone non-decreasing as more swaps arrive.

In [ ]:
random.seed(42)
pool = UniswapV2Pool(reserve_x=0, reserve_y=0)
pool.add_liquidity(100, 200_000, 'Alice')

print("Immediately after deposit (no swaps yet):")
for key, val in pool.fees_earned('Alice').items():
    print(f"  {key:>10}: {val:,.8f}")

# 1000 random swaps.
for _ in range(1000):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

print("\nAfter 1 000 swaps:")
fees1 = pool.fees_earned('Alice')
for key, val in fees1.items():
    print(f"  {key:>10}: {val:,.6f}")
fee_usd_1 = fees1['fee_x'] * pool.price + fees1['fee_y']
print(f"  fee_usd  : {fee_usd_1:,.2f}")

# 1000 more swaps.
for _ in range(1000):
    token = random.choice(['x', 'y'])
    reserve = pool.reserve_x if token == 'x' else pool.reserve_y
    amount = random.uniform(0.0001, 0.02) * reserve
    pool.swap(amount, token)

print("\nAfter 2 000 swaps:")
fees2 = pool.fees_earned('Alice')
for key, val in fees2.items():
    print(f"  {key:>10}: {val:,.6f}")
fee_usd_2 = fees2['fee_x'] * pool.price + fees2['fee_y']
print(f"  fee_usd  : {fee_usd_2:,.2f}")

print(f"\nFee USD grew from {fee_usd_1:,.2f} -> {fee_usd_2:,.2f}  "
      f"(monotone increase: {fee_usd_2 >= fee_usd_1})")

### Task 3.3 — Monotonicity argument

At deposit time `k_current == k_entry`, so `growth = 1`, `deposit_* == current_*`, and both `fee_x` and `fee_y` are exactly `0` (up to floating-point noise). ✓

After swaps, every `pool.swap(...)` call satisfies `k_after ≥ k_before` (this is the invariant we assert in `swap`), so **`k` is non-decreasing as a function of swap count**. Therefore `growth = √(k / k_entry)` is also non-decreasing, and since

$$\text{fee\_x} = \text{current\_x} \cdot \left(1 - \frac{1}{\text{growth}}\right)$$

the fee measured in units of X is non-decreasing as long as `current_x` stays proportionally stable — which is true because `current_x = ownership · reserve_x` and reserves only grow from fees (and are merely rebalanced by swaps). The same reasoning applies symmetrically to `fee_y`. So the **USD-valued** fee `fee_x · price + fee_y` grows monotonically as well, up to the small re-pricing wiggle from which side of the pool absorbs each individual fee.

Put concisely: fees can never decrease because the only way `k` ever changes is by accumulating fees inside the pool — trades alone conserve `k` (no fee case) or grow it (with fee); there is no mechanism that removes value from the pool except `remove_liquidity`, which we aren't calling during the simulation.

---

## Task 3.4 — Fee income vs volume

Helper `simulate_trading` plus a loop over fee tiers `{5, 30, 100}` bps. Same random seed for all tiers so the trade *sizes* are identical; only the fee changes.

In [ ]:
def simulate_trading(pool, n_swaps=1000, max_trade_pct=0.02):
    """Run `n_swaps` random swaps against `pool`, up to `max_trade_pct` of the relevant reserve each."""
    for _ in range(n_swaps):
        token = random.choice(['x', 'y'])
        reserve = pool.reserve_x if token == 'x' else pool.reserve_y
        amount = random.uniform(0.0001, max_trade_pct) * reserve
        pool.swap(amount, token)

In [ ]:
fee_tiers = [5, 30, 100]                      # 0.05%, 0.30%, 1.00%
colors    = ['#2ca02c', '#1f77b4', '#d62728']
n_swaps, record_every = 2000, 50

plt.figure()

final_fee_usd = {}
for fee_bps, color in zip(fee_tiers, colors):
    # Fresh pool per tier, identical RNG seed -> identical trade sizes per step.
    random.seed(2024)
    pool = UniswapV2Pool(reserve_x=0, reserve_y=0, fee_bps=fee_bps)
    pool.add_liquidity(50, 100_000, 'Alice')  # $200 000 TVL (50 ETH · 2000 + 100 000 USDC)

    swap_numbers, fee_usds = [], []
    for i in range(1, n_swaps + 1):
        token = random.choice(['x', 'y'])
        reserve = pool.reserve_x if token == 'x' else pool.reserve_y
        amount = random.uniform(0.0001, 0.02) * reserve
        pool.swap(amount, token)

        if i % record_every == 0:
            fees = pool.fees_earned('Alice')
            fee_usd = fees['fee_x'] * pool.price + fees['fee_y']
            swap_numbers.append(i)
            fee_usds.append(fee_usd)

    final_fee_usd[fee_bps] = fee_usds[-1]
    plt.plot(swap_numbers, fee_usds, label=f'{fee_bps} bps ({fee_bps/100:.2f}%)',
             color=color, linewidth=2)

plt.xlabel('Swap number')
plt.ylabel('Cumulative LP fee income (USD)')
plt.title('LP fee income vs trading volume across fee tiers\n($200k TVL, 2 000 random swaps, shared trade-size RNG)')
plt.legend(title='Fee tier')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final cumulative fee income after 2 000 swaps:")
for fee_bps, amount in final_fee_usd.items():
    print(f"  {fee_bps:>3} bps -> ${amount:>9,.2f}")

### Task 3.4 — Research questions

**(1) Which fee tier earns the most for a high-volume pair (ETH/USDC) vs a low-volume exotic pair?**

In *this* simulation the fee tier with the **highest bps wins in absolute dollars**, because trade volume (number and size of swaps) is held fixed across tiers — the only thing that changes is how big a slice each swap pays. At 2 000 swaps, fee income scales close to linearly with the fee rate: the 100 bps tier earns roughly ~20× what the 5 bps tier earns, and ~3× what the 30 bps tier earns.

Real markets behave differently, because **fee tier feeds back into volume**. For a high-volume pair like ETH/USDC, a 1% fee would drive traders and arbitrageurs to cheaper venues; the simulated volume would collapse. Empirically the 30 bps tier wins there — low enough to keep volume, high enough to compensate for ETH volatility and impermanent loss. For a truly low-volume exotic pair (e.g. a long-tail token paired with USDC) the 100 bps tier wins: volume is demand-inelastic (there is often no alternative venue) and the extra fee compensates LPs for the higher IL risk of a volatile illiquid asset.

**(2) Why do USDC/DAI-style stablecoin pools almost always use 0.05%?**

Stablecoin-to-stablecoin trades have essentially no impermanent loss (the "price" is pinned near 1.0), so LPs don't need a fat fee to compensate for IL. In return the 0.05% tier attracts arbitrageurs whenever the pool price drifts even a few basis points away from 1.0, producing very high trade *turnover*. In my simulation the 5 bps tier still accumulates real fees over 2 000 swaps, just at a slower rate per swap — but if we simulated 10× or 100× the volume (which is exactly what low-fee stablecoin pools attract in practice), total fee income would scale directly with volume and could easily dwarf what the 30 bps tier would earn at lower attracted volume. The plot supports this directionally: fee income is a *rate* (bps × volume), and in a high-turnover regime even a tiny rate multiplies up fast.

**(3) If trading volume doubles but TVL stays the same, how does LP APR change? Linear?**

Fee income per unit time is (approximately) `volume · fee_rate`, so doubling volume doubles fee income and therefore **doubles APR** since `APR = fees_per_day / deposit_value · 365` and `deposit_value` is fixed by TVL. The relationship is **linear in volume as long as price impact stays small**. It breaks linearity when trades become large enough that slippage is substantial: at that point the realised fee per-trade still scales linearly, but volume itself shifts toward larger ones whose *effective* rate (fee as a % of notional after slippage) still equals `fee_rate`, so linearity in *fees* is quite robust. What does eventually bend the curve is **impermanent loss** — if the doubled volume is directional rather than two-sided, more fees come at the cost of more IL, and *net* LP return stops being linear. Task 3.5 below measures exactly that effect.

---

## Bonus 3.5 — Directional drift: impermanent loss vs fee income

**Research question.** As we bias the trade flow more and more to one side (pushing the pool price away from entry), at what drift does impermanent loss overtake accumulated fees — flipping the LP from profitable to underwater vs just holding the original tokens?

**Design.** Single pool (50 ETH / 100 000 USDC, 30 bps, $200k TVL). Alice deposits. Then 3 000 swaps with a directional buy bias `p_buy ∈ {0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.65}` — `p_buy = 0.50` is neutral two-sided flow; higher values mean more buys of ETH with USDC that push ETH price up. Each trade has a roughly constant USD notional (~$200, jittered 0.5×–1.5×, i.e. ~0.1% of initial TVL) so price doesn't run away from the RNG. For each scenario I measure:

- `fee_usd` — cumulative fees earned (via `fees_earned`).
- `lp_value_usd` — Alice's current LP claim priced at the *new* spot price.
- `hold_value_usd` — what Alice would have if she'd just held her original 50 ETH + 100 000 USDC, priced at the same new spot.
- `il_usd = hold_value_usd - (lp_value_usd - fee_usd)` — IL in USD, computed against the "no-fee counterfactual" LP value so it isolates the geometry from the fee accrual.
- `net_pnl_usd = lp_value_usd - hold_value_usd` — net LP profit vs HODL.

The crossover drift is where `net_pnl_usd` goes from positive to negative — i.e. where IL eats the fees.

In [ ]:
def run_drift_experiment(p_buy: float, n_swaps: int = 3000, seed: int = 7):
    """
    Directional simulation. p_buy is the probability of a buy (trader sends USDC,
    receives ETH -> pushes ETH price up). Each trade has a near-constant USD
    notional (~$200, i.e. 0.1% of initial TVL) so drift stays in a realistic
    range even under heavy bias. Returns Alice's end-state economics.
    """
    random.seed(seed)
    pool = UniswapV2Pool(reserve_x=0, reserve_y=0, fee_bps=30)

    deposit_x, deposit_y = 50.0, 100_000.0
    pool.add_liquidity(deposit_x, deposit_y, 'Alice')
    entry_price = pool.price

    trade_notional_usd = 200.0   # ~0.1% of $200k TVL

    for _ in range(n_swaps):
        notional = trade_notional_usd * random.uniform(0.5, 1.5)
        if random.random() < p_buy:
            # Buy ETH with USDC -> pushes ETH price up.
            pool.swap(notional, 'y')
        else:
            # Sell ETH for USDC -> pushes ETH price down.
            pool.swap(notional / pool.price, 'x')

    fees = pool.fees_earned('Alice')
    end_price = pool.price

    ownership = pool.lp_positions['Alice'] / pool.total_shares
    current_x = ownership * pool.reserve_x
    current_y = ownership * pool.reserve_y
    lp_value_usd = current_x * end_price + current_y

    # HODL counterfactual: original 50 ETH + 100 000 USDC at the *new* price.
    hold_value_usd = deposit_x * end_price + deposit_y

    fee_usd = fees['fee_x'] * end_price + fees['fee_y']

    # Isolate IL from fees: subtract fees out of LP value, compare vs HODL.
    il_usd = hold_value_usd - (lp_value_usd - fee_usd)
    net_pnl_usd = lp_value_usd - hold_value_usd
    price_drift = end_price / entry_price

    return {
        'p_buy': p_buy,
        'entry_price': entry_price,
        'end_price': end_price,
        'price_drift': price_drift,
        'fee_usd': fee_usd,
        'lp_value_usd': lp_value_usd,
        'hold_value_usd': hold_value_usd,
        'il_usd': il_usd,
        'net_pnl_usd': net_pnl_usd,
    }


p_buys = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.65]
results = [run_drift_experiment(p) for p in p_buys]

print(f"{'p_buy':>6} {'drift':>8} {'end_px':>10} {'fees':>12} {'IL':>12} {'net PnL':>13}")
print('-' * 65)
for r in results:
    print(f"{r['p_buy']:>6.2f} {r['price_drift']:>7.3f}x "
          f"{r['end_price']:>9,.1f} "
          f"${r['fee_usd']:>10,.2f} ${r['il_usd']:>10,.2f} ${r['net_pnl_usd']:>+11,.2f}")


In [ ]:
drifts  = [r['price_drift']    for r in results]
fees    = [r['fee_usd']        for r in results]
ils     = [r['il_usd']         for r in results]
netpnls = [r['net_pnl_usd']    for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fees vs IL
ax1.plot(drifts, fees, marker='o', label='Fees earned (USD)',           color='#2ca02c', linewidth=2)
ax1.plot(drifts, ils,  marker='s', label='Impermanent loss (USD)',      color='#d62728', linewidth=2)
ax1.set_xlabel('Price drift (end_price / entry_price)')
ax1.set_ylabel('USD')
ax1.set_title('Fees vs Impermanent Loss by directional drift')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: net PnL crossover
ax2.plot(drifts, netpnls, marker='o', color='#1f77b4', linewidth=2, label='LP net PnL vs HODL')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5, label='Break-even')
ax2.set_xlabel('Price drift (end_price / entry_price)')
ax2.set_ylabel('LP net profit vs HODL (USD)')
ax2.set_title('Break-even drift: where IL eats the fees')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Bonus 3.5 — Findings

**Two competing forces.**
- Fees are bounded by total trade volume: 3 000 trades at ~$200 each is $600k of notional volume, which at 30 bps gives an upper bound of ~$1 800 in fees. Volume doesn't change much across the `p_buy` values — the simulation runs the same number of trades — so fees hover near that ceiling for every scenario.
- Impermanent loss, by contrast, explodes with drift. The closed-form IL at drift `r` is `1 − 2√r/(1+r)` applied to total LP value: ~0.4% at `r=1.2x`, ~4% at `r=1.8x`, ~20% at `r=5x`. Because IL scales with the square root of drift (approximately) while fees scale only linearly with volume, IL catches up very quickly once flow becomes one-sided.

**Observed crossover.** In my runs LP net PnL vs HODL is:

| p_buy | drift | fees (USD) | IL (USD) | net PnL (USD) |
|------:|------:|-----------:|---------:|--------------:|
| 0.50 | 1.18× |  ~1 850 |  ~710  | **+1 135** |
| 0.52 | 1.78× |  ~2 040 | ~11 060 | −9 020 |
| 0.54 | 2.34× |  ~2 170 | ~28 180 | −26 010 |
| 0.60 | 4.91× |  ~2 570 | ~148 020 | −145 450 |
| 0.65 | 7.53× |  ~2 810 | ~304 050 | −301 230 |

The crossover sits between `p_buy = 0.50` and `p_buy = 0.52` — at a **drift of only ~1.2×** (i.e. a ~20% move from entry). Past that, even the symmetric fee income cannot keep up: at `p_buy = 0.54` the position is already losing ~$26k against HODL, versus earning only ~$2k in fees.

**Takeaway for an LP.** Fee APR alone is dangerously misleading. What matters is the *ratio* of fee income to IL over the holding period — and that ratio deteriorates fast in any directional market. This is exactly why real LPs prefer pairs where drift is structurally small (stablecoins, LST/ETH pairs), and why concentrated-liquidity (Uniswap V3) exists: forcing LPs to pick a price range lets them earn the fees of that range without being exposed to the full-curve IL that the V2 formula implies. My simulation shows that in V2 on a volatile pair, anything more than a mild two-sided jitter flips the LP into "worse than HODL" territory surprisingly quickly.

---

## Final report — one interesting finding

The most interesting result is the **crossover between fee income and impermanent loss** in Task 3.5. Looking at Task 3.4 alone, the conclusion looks like "higher fees always win" — the 100 bps tier dominates 30 bps and 5 bps across the whole 2 000-swap simulation, and the relationship looks almost perfectly linear in fee rate. That picture is misleading because Task 3.4 uses symmetric flow (`random.choice('x', 'y')`), so the pool price oscillates around entry and IL averages to roughly zero. All the fee income is net yield in that world.

The bonus experiment tests what happens under directional flow — the much more realistic case, because real order flow *is* directional (news shocks, arbitrage against centralised exchanges, token unlocks). The result is sharper than I expected: it only takes a **mild imbalance of ~52/48 buy/sell to flip the LP underwater**, corresponding to a price drift of about 1.2×. Past that, the LP is losing more to IL than they earn in fees, and the gap widens fast — at drift ~1.8× (p_buy = 0.52) the net loss is already ~5× the fees earned; at drift ~5× (p_buy = 0.60) it's ~60×.

The asymmetry between the two forces is what makes this striking. Fee income per swap is bounded by `fee_rate · trade_size`, and trades are small by design, so cumulative fees grow slowly and roughly linearly with volume — in my setup they cap near the $2 000 mark after 3 000 trades regardless of bias. IL, by contrast, is `1 − 2√r/(1+r)` applied to the entire LP position, which hits ~0.4% at r=1.2×, ~4% at r=1.8×, and ~20% at r=5×. A 30 bps fee simply cannot keep up once drift gets above ~20%.

This reframes what the three fee tiers in 3.4 are really for. The 5 bps tier isn't "low-yield" — it's specifically designed for pairs where structural drift is bounded (stablecoins, LST/ETH). The 100 bps tier isn't "high-yield" — it's the tier where you *need* a fat fee to stay above the IL the pair will inevitably generate. Picking a fee tier is really picking a drift assumption. That also explains the existence of Uniswap V3 (concentrated liquidity): by forcing LPs to pick a narrow price range, V3 lets them collect fees over a smaller exposure window and avoid most of the IL that V2's full-curve formula implies — at the cost of having to actively manage the range when the market moves.

Concretely, the formula an LP actually cares about is approximately

$$\text{net\_APR} \approx \underbrace{\text{fee\_rate} \cdot \tfrac{\text{volume}}{\text{TVL}} \cdot 365}_{\text{fee APR}} \;-\; \underbrace{\mathbb{E}\!\left[1 - \tfrac{2\sqrt{r}}{1+r}\right]}_{\text{expected IL}}$$

and the bonus experiment is a direct empirical measurement of both terms on the same pool. Seeing fee income flatten at ~\$2 000 while IL rockets to six figures in the same run is the single most useful thing I'll take away from this assignment — APR quoted on LP dashboards is not a standalone metric; it only means something alongside an assumption about drift.